# Study 822 — Omega-Ratio Sort ⚖️

**Does a full gain/loss ratio beat plain trailing Sharpe?**

Keating & Shadwick (2002) sell the **Omega ratio** — `Ω(0) = E[max(r,0)] / E[max(−r,0)]`,
the ratio of a name's average gain to its average loss — as a "universal" performance
measure that reads the *whole* return distribution (every moment, both tails), unlike the
Sharpe ratio which stops at mean and variance. The pitch: sort a cross-section on trailing
Omega (long high-Omega / short low-Omega) and this richer, distribution-aware sort should
beat a plain trailing-Sharpe sort ([study 814](../../814-trailing-sharpe-anomaly/)). We put
that head-to-head on a liquid US cross-section (2010-01-04 → 2026-06-30, 50 names).

*Numbers below are the frozen headline (`docs/results.md`); the live cells run the fast
synthetic control. Survivorship: current-membership mega-caps — magnitudes are an upper
bound.*


## 1. The idea in one picture

Sharpe divides a name's average return by its volatility — two numbers, and it's blind to *shape*. The **Omega ratio** at 0 instead adds up all the up-day returns and divides by all the down-day losses: `Ω(0) = avg gain / avg loss`. It 'sees' skewness and fat tails. The claim is that this fuller picture should pick better stocks than Sharpe. We test it — and check whether Omega is really any different from Sharpe once you point both at real returns.

In [1]:
import numpy as np, pandas as pd
R = dict(omega_bps=1.19, omega_t=0.76, sharpe_bps=1.29, sharpe_t=0.83,
         rho_omega_sharpe=0.996, hi_bps=7.93, lo_bps=6.74, gross_sharpe=0.18)
print('long high-Omega / short low-Omega spread: %+.2f bps/day (NW t = %+.2f)'
      % (R['omega_bps'], R['omega_t']))
print('  vs the plain trailing-Sharpe sort       : %+.2f bps/day (NW t = %+.2f)'
      % (R['sharpe_bps'], R['sharpe_t']))
print('  Omega ~ Sharpe per-day rank correlation  : %+.3f  (near-identical sorts)'
      % R['rho_omega_sharpe'])
print('  gross spread Sharpe (before cost)        : %.2f' % R['gross_sharpe'])

long high-Omega / short low-Omega spread: +1.19 bps/day (NW t = +0.76)
  vs the plain trailing-Sharpe sort       : +1.29 bps/day (NW t = +0.83)
  Omega ~ Sharpe per-day rank correlation  : +0.996  (near-identical sorts)
  gross spread Sharpe (before cost)        : 0.18


## 2. Is the sort just noise? A live synthetic control

We plant the effect in a seeded toy world (`edge>0`: low-vol / high-Omega names really do out-earn) and check the detector recovers it — and that it stays *silent* on the null (`edge=0`, Omega varies but predicts nothing). No network.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from omega_ratio import data, strategy as st
null = st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=822, n_assets=40, n_days=1200))
planted = st.synthetic_detect(data.synthetic_panel(edge=0.0016, seed=822, n_assets=40, n_days=1500))
print('null world   : spread NW t = %+.2f  (should be ~0)' % null['t_nw'])
print('planted world: spread NW t = %+.2f  (should light up)' % planted['t_nw'])

null world   : spread NW t = +1.44  (should be ~0)
planted world: spread NW t = +9.67  (should light up)


## 3. The honest verdict — Omega does *not* beat Sharpe here

On this liquid mega-cap tape the long-high-Omega / short-low-Omega spread is **+1.19 bps/day** with NW *t* = **+0.76** — the right sign, but nowhere near the |*t*| ≥ 2 significance bar. And the head-to-head is brutal: the Omega sort is **0.996 rank-correlated with the plain Sharpe sort** and earns *less* than it (+1.29 bps). The 'whole distribution' Omega is sold on buys nothing over mean/vol here — on daily equity returns `Ω(0)` is just a re-labelling of Sharpe, and it inherits Sharpe's insignificance. The seeded synthetic control fires cleanly on a *planted* effect, so this is a genuine null, not a broken sort. **Signal: None**, **Tradability: Mirage** (the thin edge dies at any realistic cost).